#Load Environment

In [ ]:
import csv
import requests
#using beautifulesoup to scrape the site
try:
    from bs4 import BeautifulSoup
except ImportError:
    !pip install beautifulsoup4
    from bs4 import BeautifulSoup
from bs4 import BeautifulSoup
import re
# Install the necessary libraries
!pip install google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client
# Import the libraries
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

#Connect to Google Drive

In [ ]:
# Optional Colab-only setup: mount your own Google Drive if needed.
# from google.colab import drive
# drive.mount("/content/drive")


#Create path for saving scraped files

In [ ]:
from pathlib import Path
import os

# Portable output workspace; override SJK_PORTFOLIO_DIR to use a different folder.
PROJECT_DIR = Path(os.environ.get('SJK_PORTFOLIO_DIR', '/content/sjk_portfolio')) / 'Kanboard_Using_Help_File_Scrape'
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
path = str(PROJECT_DIR)


#Save URL's to CSV for scraping

In [ ]:
import csv
import requests
from bs4 import BeautifulSoup

def pagesource(url):
  # Get the page source
  response = requests.get(url)
  soup = BeautifulSoup(response.text, "html.parser")
  return soup

# Set the base URL and keyword
base_url = "https://docs.kanboard.org"
keyword = "user"

# Set the URL to scrape
url = base_url + "/v1/user/"

# Get the page source
soup = pagesource(url)

# Find all the links on the page
links = soup.find_all('a')

# Flag to track whether we are within the desired range of URLs
in_range = False

# Open the CSV file in write mode
with open(str(PROJECT_DIR / 'HelpContentStep1.csv'), 'w', newline='') as csvfile:
  # Create a CSV writer object
  writer = csv.writer(csvfile)

  # Iterate through the links
  for link in links:
    try:
      # Extract the href attribute of the link
      href = link['href']

      # Add the base URL to the URL
      full_url = base_url + href

      # Check if the URL is the first /en/ URL
      if full_url == url:
        # Set the in_range flag to True
        in_range = True
      # Check if the in_range flag is set to True and the URL is not the first /en/ URL
      elif in_range and full_url != url:
        # Check if the URL contains the keyword
        if keyword in full_url:
          # Write the URL to the CSV file
          writer.writerow([full_url])

          # Print the full URL
          print(full_url)
    except:
      # Ignore any errors that might occur
      pass

#Find and save all help file URL's from step 1

In [ ]:
import csv
import requests
from bs4 import BeautifulSoup

# Open the CSV file containing URLs
with open(str(PROJECT_DIR / 'HelpContentStep1.csv'), 'r') as f:
    reader = csv.reader(f)
    next(reader) # skip the header row

    # Loop through each row in the CSV file
    for row in reader:
        url = row[0] # the URL is in the first column

        # Make a request to the URL and get the HTML content
        response = requests.get(url)
        html = response.content

        # Parse the HTML with BeautifulSoup
        soup = BeautifulSoup(html, 'html.parser')

        # Find all the <h2> headers
        headers = soup.find_all('h2')

        # Create a new list to store the extracted text
        extracted_text = []

        # Loop through each header and extract the text between them
        for i in range(len(headers) - 1):
            # Skip the table of contents header
            if "table of contents" in headers[i].text.lower():
                continue

            # Extract the text between the headers
            start = headers[i].next_sibling
            end = headers[i+1]
            text = ""
            while start != end:
                if hasattr(start, "text"):
                    text += start.text.strip() + "\n"
                start = start.next_sibling

            # Append the extracted text to the list
            extracted_text.append(text)

        # Create a new row with the extracted text and the URL
        new_row = [url] + extracted_text

        # Append the new row to a new CSV file
        with open(str(PROJECT_DIR / 'HelpContentStep2.csv'), mode='a', newline='') as outfile:
            writer = csv.writer(outfile)
            writer.writerow(new_row)


In [ ]:
import csv
import requests
from bs4 import BeautifulSoup

# Read the URLs from the CSV file
urls = []
with open(str(PROJECT_DIR / 'HelpContentStep1.csv')) as file:
    reader = csv.reader(file)
    for row in reader:
        urls.append(row[0])

# Open each URL and extract the text
texts = []
for url in urls:
    response = requests.get(url)
    soup = BeautifulSoup(response.content, "html.parser")
    text = soup.get_text()
    texts.append(text)

# Save the URLs and texts in a new CSV file
rows = zip(urls, texts)
with open(str(PROJECT_DIR / 'HelpContentStep2.csv'), "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["URL", "Text"])
    writer.writerows(rows)

print("Texts extracted from the HTML pages and saved to texts.csv file successfully.")


#Save article content under each H2/H3 header to article column